# Notebook 04: Synthetic Task Generator

**Purpose**: Build, validate, and freeze the synthetic task generator. This is SATA's training data source and the testbed for RQ2/RQ4.

## Why synthetic tasks exist at all — the ground-truth problem

Every notebook up to this point works with real TableShift data, where we can measure accuracy but never know for certain *which features the label actually depends on* — a feature's true causal role in, say, income prediction isn't something any dataset can hand you directly. That's a hard ceiling on what real data can support: **π_true (the true feature-importance ranking) simply doesn't exist for TableShift datasets.** Without it, RQ4's success criterion — "does SATA's learned reweighting produce more *correct* reliance, not just higher accuracy?" — has nothing to check against. `src/evaluation/faithfulness_correctness.py::rank_by_true_importance` needs a ground truth to rank against, and only a generator we built ourselves can supply one.

This is also **why SATA has to be meta-trained on synthetic data rather than TableShift itself** (the spec's "data contamination guarantee"): SATA's training signal (`src/models/sata_targets.py::compute_target_scores`) is built directly from ground-truth regime/counter-spurious labels that only exist because we generated the data and know the causal rule. Training SATA on real TableShift rows would mean either fabricating that supervision (defeating the purpose) or training on accuracy alone (which is exactly the faithfulness gap RQ3 exists to expose). Keeping the two arms strictly separate is what lets a later claim like "SATA transfers to real data" (the Week-8 stretch goal referenced in Notebook 03) mean something — the model genuinely never saw TableShift rows during training.

The rule families (linear / threshold / tree / sparse-interaction) and six environments below are exactly the meta-training distribution the lit review describes for SATA (Section 3, Task 2): "meta-trained on a synthetic distribution of tabular tasks spanning linear, threshold, tree-style, and sparse-interaction rules, each instantiated under six shift types."

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

## Generator specification

See `src/data/generator.py::SyntheticTask`. Rule families: linear, threshold, tree (depth 2-3), sparse_interaction. Six environments per task: id, covariate, spurious_reversal, extrapolation, missing_feature, mechanism.

**Why four rule families, and why hold one out entirely?** A generator with only one rule family (say, linear) would let SATA learn a shortcut of its own — "always weight demos this specific way" — that happens to work because every training task shares the same functional form. Four structurally distinct families (linear combination, single/multi threshold, tree-structured regimes, multiplicative interaction) force SATA to learn something about *demonstration relevance relative to a task's regime structure* rather than memorising one rule shape. `sparse_interaction` is held out entirely from meta-training (`heldout_family` in `configs/default.yaml`) specifically so RQ4's accuracy comparison can be run on tasks whose *rule family* SATA has never seen — this is the direct analogue of Goddard et al.'s (2025) pretraining-task-diversity phase transition (Lit-review §2.3.1): below a diversity threshold, ICL-like systems produce solutions that only work on tasks resembling training; above it, they generalise across the task space. Testing on the held-out family is how this project checks which side of that threshold SATA's meta-training lands on.

**Why six environments, specifically these six?** Each is a controlled instantiation of one shift concept from the taxonomy, deliberately built so ground truth is known by construction (no DISDE decomposition needed, unlike TableShift's naturally-occurring shifts): `covariate` moves `P(x)` only; `spurious_reversal` and `mechanism` move `P(y|x)` (concept shift) via two different mechanisms — flipping which value of the spurious feature correlates with the label, vs. changing the causal rule's own coefficients/threshold; `missing_feature` and `extrapolation` are stress tests of robustness rather than points on the covariate/concept axis. `id` is the baseline every other environment is compared against.

In [2]:
from src.data.generator import SyntheticTask, generate_task_suite, RULE_FAMILIES, ENVIRONMENTS

task = SyntheticTask(
    task_id='smoke_test', rule_family='linear', causal_features=[0, 2, 4],
    coefficients=[1.0, -0.5, 2.0], spurious_strength=0.85,
)
X, y, metadata = task.generate_environment('id', n_samples=96, seed=0)
X.shape, y.mean()

((96, 10), np.float64(0.5))

## Task sampling

`generate_task_suite(config.generator)` samples `n_train_tasks` tasks (excluding the held-out family) plus `n_heldout_family_tasks` tasks of `heldout_family` only.

Four disjoint task pools exist for the same reason train/val/test splits exist anywhere: `tasks_train` is what SATA's weights are fit to (Notebook 05); `tasks_val` is what Gate 2 and early-stopping-style model selection use, so SATA isn't validated on the exact tasks it was trained on; `tasks_test` is the untouched set RQ2's protocol × shift-type grid and most of RQ4's accuracy comparison are computed on; `tasks_heldout_family` is the *sparse_interaction*-only pool used specifically to test generalisation to a rule family SATA never saw a single example of during training.

In [3]:
from src.data.generator import generate_val_test_tasks

tasks = generate_task_suite(config.generator)
train_tasks = [t for t in tasks if t.task_id.startswith('train_')]
heldout_family_tasks = [t for t in tasks if t.task_id.startswith('heldout_')]
val_tasks, test_tasks = generate_val_test_tasks(config.generator)

len(train_tasks), len(val_tasks), len(test_tasks), len(heldout_family_tasks)

(2000, 200, 200, 50)

## Validation gate (Week 2-3)

For ~50 randomly sampled tasks, two classifiers each answer a different question (see "Update 2" below for why one classifier can't answer both): an **ERM classifier** (`LogisticRegression` fit on all 10 features, including the spurious one) checks whether the shortcut actually gets exploited, and a separate **oracle classifier** (`XGBClassifier` fit on the task's causal features only) checks whether the causal mechanism genuinely changes under `mechanism`. Check:
- ERM ID accuracy > 80%
- ERM spurious-reversal accuracy < 60%
- Oracle mechanism-shift accuracy < oracle ID accuracy by >=15 points

**Gate criterion**: >=80% of sampled tasks show the expected degradation profile. Only then freeze the generator.

**What this gate is actually checking, and why it's not a formality.** The whole premise of this project — real and synthetic arms alike — is that models exploit shortcuts: features that are predictive in training but not causally load-bearing (Geirhos et al. 2020, Lit-review §2.2.1). If our synthetic tasks *don't* actually exhibit that failure mode — if a simple classifier trained on 64 demos already generalises fine across every environment — then the generator hasn't built a testbed for shortcut learning at all, it's built a testbed for something else, and every downstream number (SATA's Gate 2, RQ2's grid, RQ4's comparison) would be measuring performance on tasks that don't have the property the whole thesis is about.

This gate is a cheap, LLM-free proxy for exactly that check: ID accuracy confirms the task is learnable at all; spurious-reversal accuracy dropping below chance-adjacent levels confirms the classifier actually latched onto the spurious feature rather than the causal one; the mechanism-shift drop confirms concept shift is real, not just relabelling. Nagarajan et al.'s (2021) two-mechanism account of shortcut learning (Lit-review §2.2.1) — a geometric skew favouring low-norm spurious separators, and a statistical skew from gradient descent's convergence rate along the spurious direction — describes *why* a learner would end up here; this gate is the empirical test of whether our generator actually reproduces that pathology, not just assumed it does.

----

## Update: validator changed from XGBoost to LogisticRegression, generator rule complexity reworked

The gate above originally used `XGBClassifier(max_depth=4, n_estimators=100)` and failed at a 12% pass rate (6/50) — well short of the 80% threshold. Digging into *why* surfaced two separate problems, one about the generator and one about the gate's own methodology, and both needed fixing:

**1. The generator's `tree`/`threshold` rule families were simpler than this notebook's own spec.** The class docstring for `tree` describes "depth 2-3 nested if/else on 2-3 causal features -> 4-8 regimes", but `_apply_rule` implemented a flat 2-feature AND (2 outcomes). `threshold` was a single fixed `threshold=0.0` OR of 2 features, not the documented multi-feature combination. Both were trivially easy for *any* reasonably capable learner to recover exactly from 64 demos, which meant the classifier never needed the spurious shortcut at all — it just learned the true (easy) rule directly, and stayed accurate under `spurious_reversal` regardless of how strong the spurious feature was made (verified empirically up to spurious_strength=0.999, which only made things worse by making `mechanism`'s accuracy gap shrink instead). `src/data/generator.py` now implements `tree` as a genuine depth-3 lookup over up to 3 causal features (8 fixed leaf regimes, paired by sign-complement so the `mechanism` environment's full sign-flip reliably inverts the label), and `threshold` as a majority vote over 3 independently-thresholded features (also fixes a 75/25 class-imbalance problem the old AND/OR-of-2 had). `mechanism` itself was also changed to flip the sign of *all* causal features rather than half, since half left too small an accuracy gap on most tasks across every family.

**2. XGBoost was a mismatched choice of proxy learner for what this gate claims to test.** The gate's own theoretical citation, Nagarajan et al. (2021), demonstrates its two shortcut-learning mechanisms (geometric skew from max-margin separators, statistical skew from gradient descent) *for linear classifiers and for neural networks* — not for tree ensembles. This project's own lit review (§2.2.1) even lists "tree based models" among the standard techniques for **mitigating** exactly this pathology. Tree ensembles have a different inductive bias (axis-aligned recursive partitioning) that happens to make them the wrong instrument for the `tree`/`threshold` families specifically, independent of the generator fix above: an axis-aligned true rule is nearly free for a tree ensemble to learn, so it was never going to reach for the shortcut regardless of how the generator was tuned. Switching to `LogisticRegression` — the actual model class Nagarajan et al.'s theory is written about — gives the gate real theoretical grounding instead of an ad hoc "cheap proxy" choice. (Verified empirically: neither XGBoost nor LogisticRegression alone was a full fix without also reworking the generator — each model's blind spot tracks its own inductive bias, which is itself informative: susceptibility to a shortcut is as much a model-class property as a generator property, mirroring the point this project's lit review makes in §2.1.2 about covariate-shift correction being model-class-dependent rather than a property of the shift alone.)

Combined, these two changes took the gate from a 12% pass rate to roughly 50-55% in repeated sampling (pooled across several seeds) — a large improvement, but not yet reliably above the 80% threshold. The remaining gap is concentrated in `linear`'s spurious-reversal criterion and `tree`'s mechanism-gap criterion; further tuning of `spurious_strength_range`/`label_noise`, or further rule-complexity adjustments per family, is still needed before treating the generator as frozen. See the gate's live output below for the current numbers.

----

## Update 2: decoupled ERM/oracle criteria, Bernoulli linear labels, tree leaf exclusions (gate 34% -> 90% real run)

Update 1's fixes moved the real gate from 12% to 34% (17/50) — better, but still stuck well below 80%, and further tuning of `spurious_strength_range`/`label_noise` alone never got past ~75% even pooled over many seeds. The reason turned out to be structural, not a matter of finding the right numbers.

**Why the gate had a ceiling.** In `generate_environment`, the `mechanism` environment regenerates the spurious feature from the *new* (mechanism-shifted) label — so the spurious cue still works there just as well as in `id`. That means criterion 2 (spurious-reversal accuracy < 60%) rewards a classifier whose reliance on the spurious feature dominates its reliance on the causal features, while criterion 3 (mechanism-gap >= 15pt) rewards the *opposite* — a classifier whose causal reliance dominates — measured on the exact same trained classifier. For a linear model this is literally `|w_spurious| > |w_causal · x|` vs. the reverse inequality, on the same fitted weights: a per-task knife-edge. Every tuning attempt just traded one criterion against the other (confirmed empirically: pushing `spurious_strength_range` higher consistently improved criterion 2 while shrinking criterion 3's gap, and vice versa).

**The fix: stop asking one classifier to answer two different questions.** This is exactly the pattern Arjovsky et al. (2019, *Invariant Risk Minimization*) use to validate their own Colored MNIST benchmark (Table 1): an ERM model trained on all available signal (17.1% OOD accuracy — shows it took the shortcut) is compared against an oracle restricted to the causal signal alone (73.5%, the "grayscale" model — shows the causal signal was real and sufficient all along). Applying the same split here:
- **Criteria 1-2** (is the shortcut being exploited?) are now measured on the **ERM classifier** — `LogisticRegression` fit on all 10 features, unchanged from Update 1.
- **Criterion 3** (is there a genuine concept shift to detect?) is now measured on a separate **oracle classifier** — `XGBClassifier(max_depth=4, n_estimators=100)` fit on the task's causal features *only*. This is deliberately the reintroduction of XGBoost, but in a role where its inductive bias is a strength rather than the liability Update 1 identified: recovering a nonlinear/axis-aligned causal rule from causal features alone is exactly the kind of function a tree ensemble is good at, and Update 1's objection was specifically to using it as the *shortcut-susceptibility* detector, not as a general-purpose learner. (`src/models/sata_train.py`'s Gate 2 proxy already uses XGBoost for the same reason — a capable, fast, deterministic stand-in learner.)

**Two further generator fixes fell out of chasing the oracle numbers down:**

- **`coefficient_scale` was completely inert.** `linear`'s original label rule was `sigmoid(logit) > 0.5`, which is exactly `sign(logit)` — scale-invariant. No value of `coefficient_scale` could ever change which classifier a fixed dataset favours, so the notebook's own "adjust coefficient scale" advice (both here and in Update 1) could never have worked. Labels are now sampled `y ~ Bernoulli(sigmoid(logit))` instead of thresholded at 0.5 — the same distinction Colored MNIST relies on: a shortcut only gets learned when it is *more reliable in training* than the causal signal, which requires the causal signal to be genuinely noisy rather than perfectly, deterministically separable. `coefficient_scale` (now a real `configs/default.yaml` key, `generator.coefficient_scale`) controls how noisy: at 1.5, `linear`'s Bayes accuracy from the causal features alone is ~0.74 — deliberately below `spurious_strength_range` (0.96-0.995), so the ERM classifier has good reason to prefer the shortcut. This alone took `linear`'s criterion-2 pass rate from ~61% to 88-100% across the settings tested.
- **`tree`'s random leaf-label assignment sometimes produced degenerate functions.** Diagnosing per-task pass rates by the actual Boolean function assigned to `tree`'s 8 leaves surfaced two problem categories: "dictator" functions (the leaf labels reduce to just the sign of one single causal feature) are linearly separable, so the ERM classifier learns them directly and never needs the shortcut — criterion 2 passes only ~43% of the time on these. "Parity" functions (XOR of all 3 sign bits) are the opposite failure — not learnable from 64 samples by *any* shallow classifier, including the oracle, so criterion 3 (which needs the oracle to actually track the causal rule) passes only ~50% of the time, with oracle ID accuracy sitting near 0.57 (little better than chance). The remaining "signed majority" functions passed both criteria ~100% of the time. `_sample_tree_leaf_labels` (`src/data/generator.py`) now rejects both dictator and parity assignments when sampling a task's leaf labels.

**Net result**: a real run of this notebook (not just a pooled estimate) passes **90.0% (45/50)** — linear 81%, tree 93%, threshold 95% — comfortably clear of the 80% threshold. `data/synthetic/generator_config.json['frozen']` is `True`. Config was retuned alongside the fix: `spurious_strength_range: [0.92, 0.99] -> [0.96, 0.995]`, `label_noise: 0.08 -> 0.02`, new `coefficient_scale: 1.5`.

**One caveat for the write-up**: excluding dictator and parity leaves `tree`'s accepted leaf functions as exactly the 8 "signed majority" functions on 3 features — structurally close to `threshold`'s majority-of-3 rule (fixed thresholds at 0 with per-feature sign negation, vs. `threshold`'s independently-drawn thresholds). The two families are less structurally distinct than the original four-family design intended; a possible follow-up is `tree` using 4 causal features (16 leaves) when `n_causal >= 4`, which has a much richer space of non-degenerate functions, though its learnability from only 64 samples is untested.

A second, separate issue surfaced while tracing how these tasks get evaluated downstream: `src/evaluation/faithfulness_correctness.py`'s π_true ranking (used by Notebook 06's RQ4 correctness-of-reliance metric) ranked causal features by `|coefficients|`, which only `linear` actually reads — for `threshold`/`tree`/`sparse_interaction` the within-causal ordering was noise from an unused random vector, and non-load-bearing "causal" decoys (e.g. `threshold`'s/`tree`'s unused 4th/5th sampled causal feature) were ranked as more important than the spurious feature. Fixed alongside this update — see `src/evaluation/faithfulness_correctness.py` for the family-aware replacement.

In [4]:
import os
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


def validate_task(task, seed=0):
    """Two different classifiers answer two different questions (the IRM
    "grayscale oracle" pattern, Arjovsky et al. 2019 Table 1: ERM on all
    features vs. an oracle given only the causal ones) -- see this cell's
    markdown for why one classifier answering both broke the gate.

    ERM = LogisticRegression on all 10 features: did the classifier actually
    exploit the spurious shortcut (criteria 1-2)? Oracle = XGBoost on the
    causal features only: is the causal mechanism genuinely different under
    `mechanism`, i.e. is there a real concept shift to detect (criterion 3)?
    """
    demo_X, demo_y, _ = task.generate_environment('id', n_samples=64, seed=seed)
    causal = list(task.causal_features)

    if len(np.unique(demo_y)) < 2:
        # Degenerate demo draw (all one class) -- both estimators would raise.
        const = int(demo_y[0])
        erm_predict = lambda X: np.full(len(X), const)
        oracle_predict = lambda X: np.full(len(X), const)
    else:
        erm = LogisticRegression(max_iter=1000).fit(demo_X, demo_y)
        oracle = XGBClassifier(max_depth=4, n_estimators=100, verbosity=0, n_jobs=1).fit(demo_X[:, causal], demo_y)
        erm_predict = erm.predict
        oracle_predict = lambda X: oracle.predict(X[:, causal])

    accs, oracle_accs = {}, {}
    for env in ENVIRONMENTS:
        qX, qy, _ = task.generate_environment(env, n_samples=32, seed=seed + 1)
        accs[env] = (erm_predict(qX) == qy).mean()
        if env in ('id', 'mechanism'):
            oracle_accs[env] = (oracle_predict(qX) == qy).mean()
    return accs, oracle_accs


sample_rng = random.Random(config.seed_accuracy[0])
sampled_tasks = sample_rng.sample(train_tasks, min(50, len(train_tasks)))

try:
    n_jobs = len(os.sched_getaffinity(0))
except AttributeError:
    n_jobs = os.cpu_count() or 4

# Each task's fit+eval is independent -- LogisticRegression's lbfgs solver and
# XGBoost's C++ fit are both BLAS/native-code-backed and release the GIL, so a
# thread pool gives real parallelism across the job's allocated cores instead
# of fitting all 50 tasks (2 classifiers each) one at a time.
validation_rows = []
with ThreadPoolExecutor(max_workers=n_jobs) as pool:
    futures = {pool.submit(validate_task, task, config.seed_accuracy[0]): task for task in sampled_tasks}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Validation gate (ERM LR + causal-only XGB oracle)"):
        task = futures[future]
        accs, oracle_accs = future.result()
        passes_gate = (
            accs['id'] > 0.80
            and accs['spurious_reversal'] < 0.60
            and (oracle_accs['id'] - oracle_accs['mechanism']) >= 0.15
        )
        validation_rows.append({
            'task_id': task.task_id,
            'rule_family': task.rule_family,
            **{f'acc_{env}': accs[env] for env in ENVIRONMENTS},
            'oracle_acc_id': oracle_accs['id'],
            'oracle_acc_mechanism': oracle_accs['mechanism'],
            'passes_gate': passes_gate,
        })

validation_df = pd.DataFrame(validation_rows).sort_values('task_id').reset_index(drop=True)
pass_rate = float(validation_df['passes_gate'].mean())
gate_passed = pass_rate >= 0.80

print(f"Gate pass rate: {pass_rate:.1%} ({int(validation_df['passes_gate'].sum())}/{len(validation_df)})")
print("GATE PASSED — freezing generator." if gate_passed
      else "GATE FAILED — adjust generator parameters (spurious_strength_range, "
           "coefficient_scale, label_noise, tree leaf exclusions) and re-run this notebook.")

validation_df

Validation gate (ERM LR + causal-only XGB oracle): 100%|██████████| 50/50 [00:02<00:00, 20.63it/s]

Gate pass rate: 90.0% (45/50)
GATE PASSED — freezing generator.


,task_id,rule_family,acc_id,acc_covariate,acc_spurious_reversal,acc_extrapolation,acc_missing_feature,acc_mechanism,oracle_acc_id,oracle_acc_mechanism,passes_gate
0,train_0013,linear,0.93750,1.00000,0.43750,0.93750,0.96875,0.62500,0.81250,0.12500,True
1,train_0051,tree,0.93750,0.96875,0.31250,0.96875,0.93750,0.65625,0.96875,0.00000,True
2,train_0054,threshold,0.93750,0.84375,0.31250,0.75000,0.93750,0.75000,0.90625,0.12500,True
3,train_0061,linear,0.96875,0.93750,0.09375,0.87500,0.96875,0.93750,0.59375,0.50000,False
4,train_0065,threshold,0.96875,0.87500,0.59375,0.96875,0.90625,0.37500,0.90625,0.18750,True
5,train_0178,linear,0.96875,0.93750,0.18750,0.96875,0.96875,0.90625,0.59375,0.56250,False
6,train_0189,linear,0.96875,0.96875,0.28125,0.93750,0.96875,0.75000,0.87500,0.15625,True
7,train_0191,tree,0.96875,0.93750,0.28125,0.93750,0.96875,0.78125,0.84375,0.12500,True
8,train_0209,linear,0.90625,0.93750,0.15625,0.84375,0.96875,0.68750,0.75000,0.15625,True
9,train_0228,tree,0.96875,1.00000,0.37500,0.78125,0.93750,0.68750,0.84375,0.15625,True


In [5]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed


def save_task_environments(task, out_dir, n_demos, n_queries, seed):
    """One parquet per task: all 6 environments' demo+query rows, tagged by
    `environment`/`split` columns, plus a small metadata JSON sidecar."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for env in ENVIRONMENTS:
        X, y, metadata = task.generate_environment(env, n_samples=n_demos + n_queries, seed=seed)
        for i in range(n_demos + n_queries):
            row = {f'feature_{j}': X[i, j] for j in range(X.shape[1])}
            row.update({
                'environment': env,
                'split': 'demo' if i < n_demos else 'query',
                'label': metadata[i]['label'],
                'regime': metadata[i]['regime'],
                'is_counter_spurious': metadata[i]['is_counter_spurious'],
                'spurious_consistent': metadata[i]['spurious_consistent'],
            })
            rows.append(row)
    pd.DataFrame(rows).to_parquet(out_dir / f'{task.task_id}.parquet', index=False)

    meta = {
        'task_id': task.task_id,
        'rule_family': task.rule_family,
        'causal_features': list(int(c) for c in task.causal_features),
        'coefficients': [float(c) for c in task.coefficients],
        'spurious_strength': float(task.spurious_strength),
        'threshold': float(task.threshold),
        # 'threshold'/'tree' families only (see src/data/generator.py) --
        # None for 'linear'/'sparse_interaction'. Needed to reconstruct the
        # exact task from disk (Notebook 05's load_task_suite), since these
        # replace the flat single `threshold` those two families used to use.
        'thresholds3': [float(t) for t in task.thresholds3] if task.thresholds3 is not None else None,
        'leaf_labels': [int(v) for v in task.leaf_labels] if task.leaf_labels is not None else None,
    }
    with open(out_dir / f'{task.task_id}_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)


DATA_SYN = resolve_path(config.paths.data_synthetic)
TASK_GROUPS = {
    'tasks_train': train_tasks,
    'tasks_val': val_tasks,
    'tasks_test': test_tasks,
    'tasks_heldout_family': heldout_family_tasks,
}

try:
    n_jobs = len(os.sched_getaffinity(0))
except AttributeError:
    n_jobs = os.cpu_count() or 4

# Save regardless of gate outcome — Notebook 05/06 need *something* to develop
# and smoke-test against, and a failed gate is a signal to keep calibrating,
# not a reason to block having any data on disk. `generator_config.json`'s
# `gate_pass_rate`/`frozen` fields make the provisional status explicit; treat
# this as genuinely frozen only once `frozen` is true.
#
# Each task's generate_environment() + parquet write is independent of every
# other task, so this is embarrassingly parallel -- a thread pool (rather than
# a process pool) is used deliberately: every notebook's setup cell touches
# torch (via set_seed's torch.cuda.manual_seed_all), and fork-based process
# pools after a CUDA context has been touched are a well-known source of
# hangs/crashes, so threads are the safe choice here even though the raw
# Python dict-building loop above doesn't get full GIL-free parallelism —
# the per-task numpy work and the parquet I/O still do.
for group_name, group_tasks in TASK_GROUPS.items():
    group_dir = DATA_SYN / group_name
    with ThreadPoolExecutor(max_workers=n_jobs) as pool:
        futures = [
            pool.submit(
                save_task_environments, task, group_dir,
                config.generator.demos_per_task, config.generator.queries_per_env, config.seed_accuracy[0],
            )
            for task in group_tasks
        ]
        for _ in tqdm(as_completed(futures), total=len(futures), desc=f"Saving {group_name}"):
            pass
    print(f"{group_name}: saved {len(group_tasks)} tasks -> {group_dir}")

generator_config = {
    'n_features': config.generator.n_features,
    'n_causal_range': list(config.generator.n_causal_range),
    'rule_families': list(config.generator.rule_families),
    'heldout_family': config.generator.heldout_family,
    'spurious_strength_range': list(config.generator.spurious_strength_range),
    'label_noise': config.generator.label_noise,
    'coefficient_scale': getattr(config.generator, 'coefficient_scale', 2.0),
    'demos_per_task': config.generator.demos_per_task,
    'queries_per_env': config.generator.queries_per_env,
    'environments': ENVIRONMENTS,
    'gate_erm_classifier': 'LogisticRegression(max_iter=1000) on all features; criteria 1-2',
    'gate_oracle_classifier': 'XGBClassifier(max_depth=4, n_estimators=100) on causal features only; criterion 3',
    'tree_leaf_exclusions': ['dictator', 'parity'],
    'gate_pass_rate': pass_rate,
    'frozen': gate_passed,
}
with open(DATA_SYN / 'generator_config.json', 'w') as f:
    json.dump(generator_config, f, indent=2)

validation_df.to_parquet(DATA_SYN / 'gate_validation.parquet', index=False)

stale_artifact = DATA_SYN / 'xgboost_validation.parquet'
if stale_artifact.exists():
    stale_artifact.unlink()  # superseded first-pass gate output (XGBoost-only, pre ERM/oracle split)

if gate_passed:
    print("Generator frozen — all downstream notebooks should treat these tasks as fixed.")
else:
    print(
        f"Gate NOT passed ({pass_rate:.0%} < 80%) — data saved for development/smoke-testing "
        "Notebooks 05/06 against, but generator_config.json['frozen'] is False. See the gate "
        "markdown cell above for the current failure breakdown by rule family/criterion before "
        "treating results from Notebooks 05+ as final."
    )

Saving tasks_train: 100%|██████████| 2000/2000 [00:39<00:00, 51.26it/s]


tasks_train: saved 2000 tasks -> /home/562/cg3543/LLM-ICL-OOD-Honours/sata-project/data/synthetic/tasks_train


Saving tasks_val: 100%|██████████| 200/200 [00:03<00:00, 54.11it/s]


tasks_val: saved 200 tasks -> /home/562/cg3543/LLM-ICL-OOD-Honours/sata-project/data/synthetic/tasks_val


Saving tasks_test: 100%|██████████| 200/200 [00:03<00:00, 51.87it/s]


tasks_test: saved 200 tasks -> /home/562/cg3543/LLM-ICL-OOD-Honours/sata-project/data/synthetic/tasks_test


Saving tasks_heldout_family: 100%|██████████| 50/50 [00:00<00:00, 65.25it/s]

tasks_heldout_family: saved 50 tasks -> /home/562/cg3543/LLM-ICL-OOD-Honours/sata-project/data/synthetic/tasks_heldout_family
Generator frozen — all downstream notebooks should treat these tasks as fixed.


## Output

- `data/synthetic/tasks_train/` — 2000 task directories
- `data/synthetic/tasks_val/` — 200 tasks
- `data/synthetic/tasks_test/` — 200 tasks
- `data/synthetic/tasks_heldout_family/` — 50 tasks
- `data/synthetic/generator_config.json` — frozen generator parameters, plus `coefficient_scale`, `gate_erm_classifier`/`gate_oracle_classifier`, `tree_leaf_exclusions`
- `data/synthetic/gate_validation.parquet` — gate results, now including `oracle_acc_id`/`oracle_acc_mechanism` columns alongside the ERM `acc_{env}` columns